<a href="https://colab.research.google.com/github/CefasRepRes/shipwreck-oil-detection-colab/blob/main/CefasCobblesTools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def install_requirements():

  global np, seg, sam, predictor, unet, si, torchvision, torch, SAM2ImagePredictor, plt, build_sam2, pd, pd, cv2, measure, reload, trange, tqdm, pickle, Path, predictor, sam, device, sam_checkpoint
  from google.colab import output
  output.enable_custom_widget_manager()
  %matplotlib ipympl
  import torch
  import torchvision
  print("PyTorch version:", torch.__version__)
  print("Torchvision version:", torchvision.__version__)
  print("CUDA is available:", torch.cuda.is_available())
  import sys
  !{sys.executable} -m pip install opencv-python matplotlib
  !{sys.executable} -m pip install sam2
  !{sys.executable} -m pip install segmenteverygrain
  !{sys.executable} -m pip install rtree
  !wget https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
  from sam2.build_sam import build_sam2
  from sam2.sam2_image_predictor import SAM2ImagePredictor
  sam_checkpoint = "sam2.1_hiera_large.pt"
  device = "cuda"
  sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device)
  predictor = SAM2ImagePredictor(sam)
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import cv2
  from skimage import measure
  from importlib import reload
  import segmenteverygrain as seg
  import segmenteverygrain.interactions as si
  from tqdm import trange, tqdm
  import pickle
  from pathlib import Path
  !wget "https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt"
  # U-Net model (PyTorch)
  unet = seg.load_unet("seg_model_smooth_labels.pt")
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG
  !wget -O WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG
  !wget -O Pixel_calculations_subset.csv https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/Pixel_calculations_subset.csv

Code to create the image dropdown box

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
def Select_image():
  global IMGdropdown, min_area_px, dbs__max_dist, Roundness_Thresh
  # Folder containing the images
  image_dir = Path("/content")

  # Find JPG files, ignoring case
  image_files = sorted([
     f for f in image_dir.iterdir()
      if f.is_file() and f.suffix.lower() == ".jpg"
  ])

  # Dropdown shows filename without extension
  IMGdropdown = widgets.Dropdown(
     options=[f.stem for f in image_files],
     description="Select image:"
  )
  # Slider for pixel filter
  min_area_px = widgets.IntSlider(value=250, min=50, max=750, step=10, description='Min pixel area', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})
  dbs__max_dist = widgets.IntSlider(value=30, min=10, max=500, step=5, description='Prompt distance', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})
  Roundness_Thresh = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1, description='Roundness_threshold', readout=True, readout_format='.1f', orientation='horizontal', layout=widgets.Layout(width='400px'), style={'description_width': 'auto'})

  return display(IMGdropdown, min_area_px,dbs__max_dist, Roundness_Thresh)


Dropdown(description='Image:', options=('WWBF_CEND0121_WWBF008_STN_182_A1_006', 'WWBF_CEND0121_WWBF008_STN_182…

In [ ]:
def Run_segmentation():
  global all_grains, labels, mask_all, grain_data, fig, ax, image
  # replace this with the path to your image:
  fname = f'{IMGdropdown.value}.JPG'
  image = np.array(seg.load_img(fname))
  image_pred = seg.predict_image(image, unet, I=256)

  # decreasing the 'dbs_max_dist' parameter results in more SAM prompts
  # (and longer processing times):
  labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=dbs__max_dist.value)

  # SAM segmentation, using the point prompts from the Unet:
  all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
    sam,
    image,
    image_pred,
    coords,
    labels,
    min_area=min_area_px.value,
    plot_image=True,
    remove_edge_grains=False,
    remove_large_objects=False,
  )

In [ ]:
def save_segmented_outputs():
  global grains
  # Convert polygons to Grain objects
  grains = si.polygons_to_grains(all_grains, image=image)
  for g in tqdm(grains, desc='Measuring grains'):
    g.measure()

  pkl_filename = Path(f'{IMGdropdown.value}.JPG').with_suffix(".pkl")
  with open(pkl_filename, "wb") as f:
    pickle.dump((
            all_grains,
            labels,
            mask_all,
            grain_data,),f)
  print(f"Saved segmentation to: {pkl_filename}")

Measuring grains: 100%|██████████| 825/825 [00:00<00:00, 1765.12it/s]

Saved segmentation to: WWBF_CEND0121_WWBF008_STN_182_A1_006.pkl


In [ ]:
def interactive_cleaning():
  global plot
  # You only need to run this cell if you want to interactively edit the segmentation results and/or if you want to draw a scale bar
  # If you want to draw a scale bar, you should change the 'scale_m' parameter before running this cell

  # Set up SAM 2.1 predictor for adding new grains
  predictor = SAM2ImagePredictor(sam)
  predictor.set_image(image)

  # Display interactive interface
  plot = si.GrainPlot(
    grains,
    image = image,
    predictor = predictor,
    blit = False, # blitting does not work in Colab
    color_palette = 'tab20b',
    figsize = (12, 8),  # inches
    scale_m = 0.1,      # meters; change this before launching 'GrainPlot'
    color_by = None,
    px_per_m = 1856.6   # px/m; alternative to 'scale_m'; will be overwritten if scale bar is drawn on image, using 'scale_m'
  )

  plot.activate()

plots

In [1]:
def Plot_Auto_clean():
  from skimage.measure import regionprops, regionprops_table
  import math
  from matplotlib import colormaps
  from matplotlib.patches import Polygon as MplPolygon
  import matplotlib.pyplot as plt
  def polygon_parts(geometry):
    """
    Return individual polygon parts from a Polygon or MultiPolygon geometry.
    """

    if geometry is None or geometry.is_empty:
        return []

    if geometry.geom_type == "Polygon":
        return [geometry]

    if geometry.geom_type == "MultiPolygon":
        return list(geometry.geoms)
    return []
  def add_polygon_to_axis(
    ax,
    geometry,
    face_colour,
    edge_colour="black",
    alpha=0.55,
    linewidth=0.8):
    """
    Add a Shapely Polygon or MultiPolygon to a Matplotlib axis.
    """

    for polygon in polygon_parts(geometry):

        coordinates = np.asarray(polygon.exterior.coords)

        patch = MplPolygon(
            coordinates,
            closed=True,
            facecolor=face_colour,
            edgecolor=edge_colour,
            linewidth=linewidth,
            alpha=alpha)

        ax.add_patch(patch)
  # -------------------------------------------------------------------------
  # Calculate mask roundness from the label raster
  # -------------------------------------------------------------------------
  initial_properties = regionprops_table(
        labels.astype(int),
        properties=(
            "label",
            "area",
            "perimeter"))

  initial_mask_data = pd.DataFrame(initial_properties)

  # Calculate roundness:
  # 1.0 = perfectly circular
  # lower values = increasingly irregular / elongated
  initial_mask_data["Roundness"] = np.where(
        initial_mask_data["perimeter"] > 0, (
            4
            * math.pi
            * initial_mask_data["area"]
            / (initial_mask_data["perimeter"] ** 2)),0,)

  # -------------------------------------------------------------------------
  # Split grains at the roundness threshold
  # -------------------------------------------------------------------------
  accepted_label_ids = initial_mask_data.loc[
        initial_mask_data["Roundness"] >= ROUNDNESS_THRESHOLD,
        "label"].astype(int).tolist()

  rejected_label_ids = initial_mask_data.loc[
        initial_mask_data["Roundness"] < ROUNDNESS_THRESHOLD,
        "label"].astype(int).tolist()

  # SegmentEveryGrain label IDs correspond to polygon-list positions + 1.
  # Extract the underlying Shapely polygon from each Grain object.
  accepted_grains = [
        grains[label_id - 1].polygon
        for label_id in accepted_label_ids
        if 0 < label_id <= len(grains)]

  rejected_grains = [
        grains[label_id - 1].polygon
        for label_id in rejected_label_ids
        if 0 < label_id <= len(grains)]

  print(f"Total masks: {len(initial_mask_data)}")
  print(f"Accepted masks (roundness >= {ROUNDNESS_THRESHOLD}): "
        f"{len(accepted_label_ids)}")
  print(f"Rejected masks (roundness < {ROUNDNESS_THRESHOLD}): "
        f"{len(rejected_label_ids)}")

  # -------------------------------------------------------------------------
  # Create accepted-only and rejected-only label rasters
  # -------------------------------------------------------------------------

  labels_filtered = np.where(
        np.isin(labels, accepted_label_ids),
        labels,
        0).astype(int)

  labels_rejected = np.where(
        np.isin(labels, rejected_label_ids),
        labels,
        0).astype(int)
  # =========================================================================
  # IMAGE 1
  # Accepted grains: roundness >= threshold
  # =========================================================================
  accepted_plot_output = (f"{fname}_roundness_accepted.png")
  fig, ax = plt.subplots(figsize=(15, 10))
  # Original image
  ax.imshow(image)
  paired_colours = colormaps["Paired"]
  # Draw accepted SegmentEveryGrain polygons
  for polygon_number, polygon in enumerate(accepted_grains):

        colour = paired_colours(
            polygon_number % paired_colours.N)
        add_polygon_to_axis(
            ax=ax,
            geometry=polygon,
            face_colour=colour,
            edge_colour="black",
            alpha=0.55,
            linewidth=0.8)
  # -------------------------------------------------------------------------
  # Major/minor axes and centroid for accepted grains
  # -------------------------------------------------------------------------

  accepted_regions = regionprops(labels_filtered)

  for region in accepted_regions:

        centroid_y, centroid_x = region.centroid
        orientation = region.orientation

        minor_x = (
            centroid_x
            + math.cos(orientation)
            * 0.5
            * region.minor_axis_length)

        minor_y = (
            centroid_y
            - math.sin(orientation)
            * 0.5
            * region.minor_axis_length)

        major_x = (
            centroid_x
            - math.sin(orientation)
            * 0.5
            * region.major_axis_length)
        major_y = (
            centroid_y
            - math.cos(orientation)
            * 0.5
            * region.major_axis_length)

        # Minor axis
        ax.plot(
            (centroid_x, minor_x),
            (centroid_y, minor_y),
            color="black",
            linewidth=1)

        # Major axis
        ax.plot(
            (centroid_x, major_x),
            (centroid_y, major_y),
            color="black",
            linewidth=1)

        # Centroid
        ax.plot(
            centroid_x,
            centroid_y,
            marker="o",
            markersize=2.5,
            color="black")

  ax.set_xlim(0, image.shape[1])
  ax.set_ylim(image.shape[0], 0)
  fig.tight_layout()
  fig.savefig(
        accepted_plot_output,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0,)

  plt.close(fig)
  print(
        f"Accepted-mask plot saved to: "
        f"{accepted_plot_output}")

  # =========================================================================
  # IMAGE 2
  # Rejected grains: roundness < threshold
  # =========================================================================

  rejected_plot_output = (f"{fname}_roundness_rejected.png")
  fig, ax = plt.subplots(figsize=(15, 10))
  # Original image
  ax.imshow(image)

  # Draw rejected polygons in red
  for polygon in rejected_grains:
        add_polygon_to_axis(
            ax=ax,
            geometry=polygon,
            face_colour="red",
            edge_colour="darkred",
            alpha=0.45,
            linewidth=1.0)

  # -------------------------------------------------------------------------
  # Centroid markers on rejected grains
  # -------------------------------------------------------------------------
  rejected_regions = regionprops(labels_rejected)
  for region in rejected_regions:
        centroid_y, centroid_x = region.centroid
        ax.plot(
            centroid_x,
            centroid_y,
            marker="o",
            markersize=2.5,
            color="darkred")
  ax.set_xlim(0, image.shape[1])
  ax.set_ylim(image.shape[0], 0)
  fig.tight_layout()
  fig.savefig(
        rejected_plot_output,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0)

  plt.close(fig)
  print(
        f"Rejected-mask plot saved to: "
        f"{rejected_plot_output}")
  from PIL import Image
  import matplotlib.pyplot as plt
  # -------------------------------------------------------------------------
  # Display accepted and rejected plots side by side in Google Colab
  # -------------------------------------------------------------------------

  accepted_image = Image.open(accepted_plot_output)
  rejected_image = Image.open(rejected_plot_output)
  fig, axes = plt.subplots(2, 1, figsize=(20, 14))
  axes[0].imshow(accepted_image)
  axes[0].set_title(
    f"Accepted grains (Roundness >= {ROUNDNESS_THRESHOLD})",
    fontsize=14)
  axes[0].axis("off")
  axes[1].imshow(rejected_image)
  axes[1].set_title(
    f"Removed grains (Roundness < {ROUNDNESS_THRESHOLD})",
    fontsize=14)
  axes[1].axis("off")

  plt.tight_layout()
  plt.show()

NameError: name 'regionprops_table' is not defined